# Merging Event Lists From Multiple Obs IDs
<hr style="border: 2px solid #f5bf03" />

- **Description:** Merging event lists from multiple Obs IDs to create a mosaiced image.
- **Level:** Intermediate
- **Data:** XMM observations of M 82 (obsid=Multiple)
- **Requirements:** Must be run using pySAS version 2.3.0 or higher.
- **Credit:** Ryan Tanner (January 2026)
- **Support:** <a href="https://heasarc.gsfc.nasa.gov/docs/xmm/xmm_helpdesk.html">XMM Newton GOF Helpdesk</a>
- **Last verified to run:** 25 Janurary 2026, for SAS v22.1 and pySAS v2.3.0

<hr style="border: 2px solid #f5bf03" />

## 1. Introduction

In this tutorial we will search the XMM archive for all observations targeted at the galaxy M82. We will download a few relevant PPS files from each observation and then determine which of observations have usable data. Some observations are essentially entirely useless due to solar flares. Then from the usuable observations we will filter the event lists and then merge the event lists to form a composite image of the hot gas outflow from M 82.

There are some warnings and caveats to the merging process that are noted below in section 5.

#### SAS Tasks to be Used

- `merge`[(Documentation for merge)](https://xmm-tools.cosmos.esa.int/external/sas/current/doc/merge/index.html "merge Documentation")

#### Useful Links

- [`pysas` Documentation](https://xmm-tools.cosmos.esa.int/external/sas/current/doc/pysas/index.html "pysas Documentation")
- [`pysas` on GitHub](https://github.com/XMMGOF/pysas)
- [Common SAS Threads](https://www.cosmos.esa.int/web/xmm-newton/sas-threads "SAS Threads")
- [Users' Guide to the XMM-Newton Science Analysis System (SAS)](https://xmm-tools.cosmos.esa.int/external/xmm_user_support/documentation/sas_usg/USG/SASUSG.html "Users' Guide")
- [The XMM-Newton ABC Guide](https://heasarc.gsfc.nasa.gov/docs/xmm/abc/ "ABC Guide")
- [XMM Newton GOF Helpdesk](https://heasarc.gsfc.nasa.gov/docs/xmm/xmm_helpdesk.html "Helpdesk") - Link to form to contact the GOF Helpdesk.

<div class="alert alert-block alert-warning">
    <b>Warning:</b> By default this notebook will place observation data files in your default <tt>data_dir</tt> directory. Make sure pySAS has been configured properly.
</div>

In [ ]:
# pySAS imports
import pysas
from pysas import MyTask
pysas.sas_cfg.set_setting('pysas_verbosity','WARNING')

# HEASoftpy import
import heasoftpy as hsp

# Generic VO access routines
import pyvo as vo

# Useful imports
import os, re, shutil

# Imports for plotting
import astropy
import matplotlib.pyplot as plt
from astropy.visualization import astropy_mpl_style
from astropy.coordinates import SkyCoord
from astropy.io import fits
from astropy.wcs import WCS
from astropy.table import Table
plt.style.use(astropy_mpl_style)

# To handle certain warnings
import warnings
warnings.filterwarnings('ignore')

Filenames to be used in this notebook.

In [ ]:
instruments = ['EPN','EMOS1','EMOS2']

filtered_evtls      = {}
gti_file            = {}
time_filtered_evtls = {}
merged_event_lists  = {}
time_filt_image     = {}

for inst in instruments:
    filtered_evtls[inst]      = f'{inst}_filtered_event_list.fits'
    gti_file[inst]            = f'{inst}_gti_rate.fits'
    time_filtered_evtls[inst] = f'{inst}_time_filtered_event_list.fits'
    merged_event_lists[inst]  = f'{inst}_merged_event_list.fits'
    time_filt_image[inst]     = f'{inst}_time_filt_image.fits'

## 2. Download PPS Files

This next cell will do a `TAP` search for XMM observations within 0.1 degrees of M82. It will also only return observations where the MOS and pn were in Full Frame mode with the medium filter. This will return a list of 11 Obs IDs that we can work with. The tutorial [Using PyVO to Find Observations for Analysis](./misc-xmm-using-pyvo-to-find-obsids.ipynb) explains in depth how this query was put together.

In [ ]:
m82_pos = SkyCoord.from_name('m82')
heasarc_tap = vo.regsearch(servicetype='tap',keywords=['heasarc'])[0]
query = """SELECT obsid 
           FROM xmmmaster as cat 
           WHERE cat.mos1_mode LIKE 'FF-ME%' and 
                 cat.mos2_mode LIKE 'FF-ME%' and 
                 cat.pn_mode LIKE 'FF-ME%' and 
                 contains(point('ICRS',cat.ra,cat.dec),circle('ICRS',{},{},0.1))=1
        """.format(m82_pos.ra.deg, m82_pos.dec.deg)
obsid_table = heasarc_tap.search(query).to_table()

obsids = []
for row in obsid_table:
    obsids.append(row['obsid'])

If you uncomment the lines in the next cell, you can erase all of the data for the Obs IDs in the list above. This will "reset" the associated Obs ID directories in your data directory.

In [ ]:
# for obsid in obsids:
#     my_pps = pysas.PPSFiles(obsid, output_to_terminal=False)
#     my_pps.clear_obs_dir()

We will be using the `Pipeline Processing System` (`PPS`) files for this tutorial. Each Obs ID can have anywhere between about a dozen PPS files to a few thousand PPS files. Rather than download **ALL** of the potentially thousands of PPS files, we will only download the four files that we need.

This next cell does two things:

1. Collects the PPS filenames of files we will need.
2. Downloads **only** the PPS files we need.

To get the filenames we need, we first use the function `get_list_of_all_filenames` which returns a list of all PPS filenames for a single Obs ID *without downloading any files*. 

Then we use the function `return_filenames_from_product_dict` which returns the filenames of specific PPS product types based on the list of PPS filenames we procured previously.

Once these filenames are collected into a single list they are passed to the `download_PPS_data` function to download just the PPS files we need (Note: The input parameter `filename` can take a single filename, or a list of filenames).

In [ ]:
for obsid in obsids:
    print(f'Downloading event lists for Obs ID: {obsid}')
    my_pps = pysas.PPSFiles(obsid)
    list_pps_files  = my_pps.get_list_of_all_filenames()
    calind_file     = my_pps.return_filenames_from_product_dict(my_pps.Obs_products['CALIND_FIT'],  list_of_files=list_pps_files)
    mos_event_lists = my_pps.return_filenames_from_product_dict(my_pps.EPIC_products['MIEVLI_FIT'], list_of_files=list_pps_files)
    pn_event_lists  = my_pps.return_filenames_from_product_dict(my_pps.EPIC_products['PIEVLI_FIT'], list_of_files=list_pps_files)
    light_curves    = my_pps.return_filenames_from_product_dict(my_pps.EPIC_products['FBKTSR_FIT'], list_of_files=list_pps_files)
    list_of_files   = calind_file + mos_event_lists + pn_event_lists + light_curves
    my_pps.download_PPS_data(filename=list_of_files)

## 3. Quick Look at the Data

We will now do a quick look at the data by first looking at basic images created from the event lists, and then looking at the light curves.

In [ ]:
for obsid in obsids:
    my_pps = pysas.PPSFiles(obsid, output_to_terminal=False)
    for event_list in my_pps.EPIC_event_lists:
        with fits.open(event_list) as hdu:
            inst = hdu[0].header['INSTRUME']
        my_pps.quick_eplot(event_list, title=f'{inst} Image for Obs ID: {obsid}', image_file=f'{inst}_image.fits')

The light curves that come with the PPS files have been partially processed. The brightest sources have been removed and the curves show the in field of view events, covering the 0.5-7.5 keV band. The file contains an optimum background rate-cut threshold which we can use as a guide for whether or not there is any useful data in the observation. If the background rate-cut threshold is above 20 counts/s for the pn and 6 counts/s for both MOS then we can assume that there is no useful data for that Obs ID. The values of 20 and 6 counts/s were chosen after inspecting the data and assuming a reasonable cut off value for this target.

In [ ]:
rate_cuts  = {}
ltcv_files = {}

for obsid in obsids:
    my_pps       = pysas.PPSFiles(obsid, output_to_terminal=False)
    light_curves = my_pps.return_filenames_from_product_dict(my_pps.EPIC_products['FBKTSR_FIT'])
    rate_cuts[obsid]  = {}
    ltcv_files[obsid] = {}
    for light_curve in light_curves:
        with fits.open(light_curve) as hdu:
            inst     = hdu[0].header['INSTRUME']
            rate_cut = hdu[1].header['FLCUTTHR']
        rate_cuts[obsid][inst]  = rate_cut
        ltcv_files[obsid][inst] = light_curve
        good_rate = 'Yes'
        if inst == 'EPN':
            if rate_cut > 20:
                good_rate = 'No'
        elif inst == 'EMOS1':
            if rate_cut > 6:
                good_rate = 'No'
        elif inst == 'EMOS2':
            if rate_cut > 6:
                good_rate = 'No'
        ts = Table.read(light_curve,hdu=1)
        fig, ax = plt.subplots()
        ax.plot(ts['TIME'],ts['RATE'])
        ax.axhline(y=rate_cut, color='b')
        ax.set_yscale('log')
        ax.set_xlabel('Time (s)')
        ax.set_ylabel('Count Rate (ct/s)')
        ax.set_title(f'{inst} Light Curve for Obs ID: {obsid}\nRate Cut = {rate_cut} : Good? {good_rate}')
        fig.show()

By scanning over the light curves we see that all of them have some level of contamination from solar flares, while a few of them are potentially completely contaminated (e.g. 0657800101, 0657801701, and 0560181301). We will have to filter the data and see how much of each Obs ID contains useful data.

## 4. Filtering the Data

We start by applying a basic filter to the event lists. This will fix some of the contamination. But then we will have to create good time interval (GTI) files to filter out times of high solar activity. This will leave us with an estimate of how much useful data each Obs ID contains.

In [ ]:
def filter_event_list(in_event_list,
                      out_event_list,
                      pi_min,
                      pi_max):

    with fits.open(in_event_list) as hdu:
        inst = hdu[0].header['INSTRUME']

    if inst == 'EPN':
        filter = 'XMMEA_EP'
        pattern = 4
    elif 'EMOS' in inst:
        filter = 'XMMEA_EM'
        pattern = 12

    # Filter expression
    expression = '(PATTERN in [0:{pattern}])&&(PI in [{pi_min}:{pi_max}])&&(FLAG == 0)&&#{filter}'.format(filter=filter,pattern=pattern,pi_min=pi_min,pi_max=pi_max)

    inargs = {'table'           : in_event_list, 
              'withfilteredset' : 'yes', 
              "expression"      : expression, 
              'filteredset'     : out_event_list, 
              'filtertype'      : 'expression', 
              'keepfilteroutput': 'yes', 
              'updateexposure'  : 'yes', 
              'filterexposure'  : 'yes'}
    
    MyTask('evselect', inargs).run()

In [ ]:
for obsid in obsids:
    print(f'Obs ID: {obsid}')
    my_pps = pysas.PPSFiles(obsid, output_to_terminal=False)
    for event_list in my_pps.EPIC_event_lists:
        with fits.open(event_list) as hdu:
            inst = hdu[0].header['INSTRUME']
        filter_event_list(event_list,filtered_evtls[inst],500,10000)

In [ ]:
def apply_gti(light_curve_file,in_event_list,gti_rate_file,out_event_list):
    with fits.open(light_curve_file) as hdu:
        inst     = hdu[0].header['INSTRUME']
        obsid    = hdu[0].header['OBS_ID']
        rate_cut = hdu[1].header['FLCUTTHR']

    good_rate = True
    if inst == 'EPN':
        if rate_cut > 20:
            good_rate = False
    elif inst == 'EMOS1':
        if rate_cut > 6:
            good_rate = False
    elif inst == 'EMOS2':
        if rate_cut > 6:
            good_rate = False

    useful_data = False

    if good_rate:
        inargs = {'table'      : light_curve_file, 
                  'gtiset'     : gti_rate_file,
                  'timecolumn' : 'TIME', 
                  "expression" : "'(RATE <= {0})'".format(rate_cut)}
        
        MyTask('tabgtigen', inargs, output_to_terminal=False).run()

        with fits.open(gti_rate_file) as hdu:
            ontime = hdu[1].header['ONTIME']
        print(f'{obsid}-{inst:<5} : On Time = {ontime}')
        if ontime > 500:
            useful_data = True
            inargs = {'table'           : in_event_list,
                      'withfilteredset' : 'yes', 
                      "expression"      : "'GTI({0},TIME)'".format(gti_rate_file), 
                      'filteredset'     : out_event_list,
                      'filtertype'      : 'expression', 
                      'keepfilteroutput': 'yes',
                      'updateexposure'  : 'yes', 
                      'filterexposure'  : 'yes'}
            
            MyTask('evselect', inargs, output_to_terminal=False).run()

    return useful_data

We will now filter the event lists based on `Rate`. This will remove contamnation from solar flares. Some Obs IDs are entirely contaminated and will have no useable data after filtering. We will make a new list of Obs IDs with useful data using the cutoff that there must be at least 500 seconds of usable data.

<div class="alert alert-block alert-info">
    <b>Note:</b> We are making some general assumptions about what is a "good" rate for filtering for each instrument for all Obs IDs. A more thorough analysis would have to determine the correct rate for <i>each</i> Obs ID for <i>each</i> instrument.
</div>

In [ ]:
useful_obsids = {}
good_event_lists = {}
for inst in instruments: good_event_lists[inst] = []

for obsid in obsids:
    useful_obsids[obsid] = {}
    my_pps = pysas.PPSFiles(obsid, output_to_terminal=False)
    light_curves = my_pps.return_filenames_from_product_dict(my_pps.EPIC_products['FBKTSR_FIT'])
    for inst in instruments:
        useful_obsids[obsid][inst] = False
        useful_data = apply_gti(ltcv_files[obsid][inst],filtered_evtls[inst],gti_file[inst],time_filtered_evtls[inst])
        if useful_data:
            useful_obsids[obsid][inst] = True
            good_event_lists[inst].append(os.path.abspath(time_filtered_evtls[inst]))
    my_pps.resolve_obs_dir()

## 5. Merging Event Lists

Now that the event lists are filtered for solar flares let's take a look at the fitlered event lists.

In [ ]:
for obsid,_ in useful_obsids.items():
    my_pps = pysas.PPSFiles(obsid, output_to_terminal=False)
    for inst in instruments:
        if useful_obsids[obsid][inst]:
            my_pps.quick_eplot(time_filtered_evtls[inst], title=f'{inst} Image for Obs ID: {obsid}', image_file=time_filt_image[inst])

Taking a look at the images we see that the MOS event lists look OK now, but all of the pn event lists have out-of-time contamination (i.e. the bright stripe going out from the center). This is something that we can fix, but unfortunately it would take 4-5 hours to reprocess the data for the 8 good Obs IDs to fix the problem. For this tutorial we will just have to skip over the pn event lists for now, but this processing will be included at the end of this notebook.

We will start by merging event lists for each EPIC camera from each Obs ID with the corresponding event list from the other Obs IDs. This will create combined event lists for each EPIC camera (except for the pn, which we are not dealing with right now). To do this we will use the SAS task `merge`.

We will use a "dummy" Obs ID to create a sub-directory in our data directory where we will write the merged event lists.

In [ ]:
dummy_obsid = '0000000000'
my_obs = pysas.ObsID(dummy_obsid)
my_obs.make_work_dir()
os.chdir(my_obs.work_dir)

<div class="alert alert-block alert-warning">
    <b>Warning:</b> This proceedure comes with a number of caveats and warnings. The merged event list can be used to create an image, <b>BUT</b> its usfulness as a science product is limited. Below are some warning taken directly from the documentation for the <tt>merge</tt> task.
</div>

- Care should be taken in using any output. Cases have been found whereby new output file attributes need to be calculated on the basis of input file attributes, and no doubt, these cases will continue to crop up. In many cases, care will be needed as regards any subsequent use of a particular merged file.

- The task is now able to correct both individual input event files for small errors in attitude, prior to merging. Very thorough testing of this feature has not yet been performed. Position angle is measured clockwise on the sky.

- Problems can occur when one of the input files contains GTI extensions (not STDGTI) and the other one doesn't. Everything seems ok but evselect can't handle this case and will calculate the wrong exposure time for files extracted from the merged event file.

- The GTI merge assumes that only one set of GTIs exist in each input file. If more than one set exist then the final result will probably be wrong.

- In the event file merging, the BADPIX extensions are not merged.

- It may be that the default imagesize may change, as merged images are likely to cover a large area of the sky than single event file images.

- Out-of-time events modeling in task `esplinemap` does not work with merged eventsets.

- When merging two files (C, A say) from different exposures, the two files will be automatically ordered so that they are merged in time order (AC). If one then wants to merge a third file (B) to the first two, this third file data being from a time in-between the first two, then problems may exist with subsequent tasks performed on the resultant merged file (ACB) as the data will not be time sorted. It is advised therefore to merge all data in time order (i.e. first A with B, then AB with C).

- It is advisable also, because of the large file sizes involved, the time-consuming nature of the reprojections and the fact, in merging e.g. four event files together, that three merge runs are necessary (A+B, AB+C, ABC+D, done preferably, remember, in time order, and not for instance in file size order), that the event files are first filtered to only the good events one is interested in.

<div class="alert alert-block alert-info">
    <b>Note:</b> We did <b>NOT</b> merge the event lists in time order, thus the final event list is <i>not</i> time sorted.
</div>

It is left as an exercise to the interested user to merge the event lists in time order. Hint: The observation time is in the header of the event lists.

In [ ]:
inst = 'EMOS1'

temp_fits = 'temp.fits'

inargs = {'set1'   : good_event_lists[inst][0],
          'set2'   : good_event_lists[inst][1],
          'outset' : temp_fits,
          'mergedifferentobs' : True}

MyTask('merge', inargs).run()

for in_event_list in good_event_lists[inst][2:]:
    inargs = {'set1'   : in_event_list,
              'set2'   : temp_fits,
              'outset' : merged_event_lists[inst],
              'mergedifferentobs' : True}
    
    MyTask('merge', inargs).run()
    
    os.remove(temp_fits)
    shutil.copyfile(merged_event_lists[inst], temp_fits)

In [ ]:
my_obs.quick_eplot(merged_event_lists[inst],vmin=0.1,vmax=1000.0)

In [ ]:
inst = 'EMOS2'

temp_fits = 'temp.fits'

inargs = {'set1'   : good_event_lists[inst][0],
          'set2'   : good_event_lists[inst][1],
          'outset' : temp_fits,
          'mergedifferentobs' : True}

MyTask('merge', inargs).run()

for in_event_list in good_event_lists[inst][2:]:
    inargs = {'set1'   : in_event_list,
              'set2'   : temp_fits,
              'outset' : merged_event_lists[inst],
              'mergedifferentobs' : True}
    
    MyTask('merge', inargs).run()
    
    os.remove(temp_fits)
    shutil.copyfile(merged_event_lists[inst], temp_fits)

In [ ]:
my_obs.quick_eplot(merged_event_lists[inst],vmin=0.1,vmax=1000.0)

In [ ]:
merged_mos_event_list = 'MOS_merged_event_list.fits'

inargs = {'set1'   : merged_event_lists['EMOS1'],
          'set2'   : merged_event_lists['EMOS2'],
          'outset' : merged_mos_event_list,
          'mergedifferentobs' : True}

MyTask('merge', inargs).run()

In [ ]:
my_obs.quick_eplot(merged_mos_event_list,title='Merged MOS Image',vmin=0.1,vmax=2000.0)

We can clean up the image a little by removing the high energy x-rays and just keep the events below 2 keV. You can apply better filtering, but ideally filtering should start over with the event lists before merging and then merging after the refined filtering.

In [ ]:
out_event_list = 'MOS_merged_event_list_1_45.fits'

expression = '(PI in [{pi_min}:{pi_max}])'.format(pi_min=500,pi_max=2000)

inargs = {'table'           : merged_mos_event_list, 
          'withfilteredset' : 'yes', 
          "expression"      : expression, 
          'filteredset'     : out_event_list, 
          'filtertype'      : 'expression', 
          'keepfilteroutput': 'yes', 
          'updateexposure'  : 'yes', 
          'filterexposure'  : 'yes'}

MyTask('evselect', inargs).run()

In [ ]:
my_obs.quick_eplot(out_event_list,title='Merged MOS Image',vmin=0.1,vmax=500.0)

## 6. Clean PN Images

In this section the pn event lists are cleaned from out-of-time contamination. The final products are cleaned images, *not* event lists. This section doesn't merge the resulting images.

In [ ]:
good_event_lists

In [ ]:
useful_obsids

<div class="alert alert-block alert-info">
    <b>Note:</b> The next few cells will process the pn data to clean up the out-of-time contamination. For the 8 Obs IDs with useful data the next cell will take approximately 2-4 hours to run. It is left as an exercise to the interested user to optimize and parallelize this analysis.
</div>

In [ ]:
inst = 'EPN'
temp_fits = 'temp.fits'

for obsid, good_obs in useful_obsids.items():

    if not good_obs[inst]: continue
    
    my_obs = pysas.ObsID(obsid)
    
    inargs = {'withoutoftime' : True,
              'options'       : '-V 1'}
    
    my_obs.basic_setup(overwrite    = False,
                       rerun        = True,
                       run_emproc   = False,
                       run_rgsproc  = False,
                       odfingest_opts = {'options':'-V 1'},
                       epproc_args    = inargs)
    
    my_obs.find_event_list_files(print_output=False)

    for filename in my_obs.files['PNevt_list']:
        if re.search('.*EPN.*ImagingEvts.ds',filename):
            outoftime_file = filename

    filtered_oot_file = 'PN_clean_outoftime_file.fits'
    
    filter_event_list(outoftime_file,filtered_oot_file,500,10000)

    pn_oot_image    = 'PN_OoT_image.fits'
    pn_oot_rescaled = 'PN_OoT_image_rescaled.fits'
    pn_clean_image  = 'PN_observation_clean_image.fits'

    my_obs.quick_eplot(filtered_oot_file, image_file=pn_oot_image, vmin=1.0, vmax=1000.0)

    hsp.farith(infil1   = pn_oot_image,
               infil2   = 0.063,
               outfil   = pn_oot_rescaled,
               ops      = 'MUL',
               noprompt = True,
               clobber  = True)

    hsp.farith(infil1   = time_filt_image[inst],
               infil2   = pn_oot_rescaled,
               outfil   = temp_fits,
               ops      = 'SUB',
               noprompt = True,
               clobber  = True)

    with fits.open(time_filt_image[inst]) as src:
            source_header = src[0].header.copy()  # Copy header
    with fits.open(temp_fits) as tgt:
            target_data = tgt[0].data
    # Create new Primary HDU with target data and source header
    new_hdu = fits.PrimaryHDU(data=target_data, header=source_header)
    # Write to output file (overwrite if exists)
    new_hdu.writeto(pn_clean_image, overwrite=True)

In [ ]:
inst = 'EPN'
clean_pn_images = []
for obsid, good_obs in useful_obsids.items():
    if not good_obs[inst]: continue
    my_obs = pysas.ObsID(obsid, output_to_terminal=False)
    pn_clean_image  = 'PN_observation_clean_image.fits'
    my_obs.quick_implot(pn_clean_image, title=f'{inst} Image for Obs ID: {obsid}')
    clean_pn_images.append(os.path.abspath(pn_clean_image))

In [ ]:
clean_pn_images